In [20]:
import json
import re
from pathlib import Path
from PIL import Image
from IPython.display import display
from datetime import datetime, timedelta
from collections import deque
from datasets import Dataset, Features, Image as HFImage, Value

In [ ]:
# Put screenshots into a list of [datetime, path_to_image]
# Make sure the list is sorted by datetime

# Directory path
screenshots_dir = Path('../mining/logs/screenshots')

# Regular expression to match timestamp patterns like 20250424_173019_991476 in the screenshot name
timestamp_pattern = re.compile(r'\d{8}_\d{6}_\d{6}')

# Store (datetime, path) pairs
images_with_timestamps = []

for image_path in screenshots_dir.iterdir():
    # if image_path.is_file() and image_path.suffix.lower() in ('.png', '.jpg', '.jpeg', '.gif'):
    if image_path.is_file() and image_path.suffix.lower() in ('.jpg'):
        # Extract filename
        filename = image_path.name

        # Search for the timestamp
        match = timestamp_pattern.search(filename)
        if match:
            timestamp_str = match.group()
            # print(f"Image: {filename} | Timestamp: {timestamp_str}")

            # Convert the string to a datetime object
            dt = datetime.strptime(timestamp_str, '%Y%m%d_%H%M%S_%f')
            # print(f"Datetime Object: {dt}")

            images_with_timestamps.append((dt, image_path))

            # Open and display image
            # img = Image.open(image_path)
            # display(img)
        else:
            print(f"No timestamp found in {filename}")

# Sort the list in-place by datetime
images_with_timestamps.sort(key=lambda x: x[0])

print("number of screenshots: ", len(images_with_timestamps))

In [ ]:
# Go through all sanitized json logs, find click events, and match them with the
# screenshot right before and add session prompts
# store everything in a Hugging Face dataset

sanitized_dir = Path('../mining/logs/sanitized_json')
session_prompts_file = Path('../mining/logs/session_prompts.log')

# Process all JSON files in the directory
all_matches = []

# Set the maximum allowed time difference (1 second)
MAX_TIME_DIFF_SECONDS = 1.0
# Set the maximum number of historical items to include
MAX_HISTORY_ITEMS = 5

# Load session prompts from log file
session_prompts = []
with open(session_prompts_file, 'r') as f:
    for line in f:
        # Parse timestamp and prompt from each line
        # Format: YYYY‑MM‑DD HH:MM:SS  Prompt text
        if line.strip():
            timestamp_str = line[:19]  # Extract the timestamp part
            prompt = line[21:].strip()  # Extract the prompt part

            # Convert timestamp string to datetime object
            timestamp = datetime.strptime(timestamp_str, '%Y‑%m‑%d %H:%M:%S')

            session_prompts.append({
                'timestamp': timestamp,
                'prompt': prompt
            })

print(f"Loaded {len(session_prompts)} session prompts from log file")

# Sort session prompts by timestamp
session_prompts.sort(key=lambda x: x['timestamp'])

# Sort JSON files by name (which contains timestamps) to process them in chronological order
json_files = sorted([f for f in sanitized_dir.iterdir() if f.is_file() and f.suffix.lower() == '.json'])

# We'll process all JSON files together to ensure proper time-based matching
all_events = []
all_click_events = []
all_keyboard_events = []

# Load all events from all files first
for json_path in json_files:
    print(f"Loading events from {json_path.name}...")

    # Load the JSON file
    with open(json_path, 'r') as f:
        data = json.load(f)

    # Process events
    for event in data['events']:
        # Add source file information
        event["source_file"] = json_path.name

        # Add to all events list
        all_events.append(event)

        # Add to specific event type lists
        if event['event'] == 'MOUSE' and event['event_type'] == 'SINGLE_CLICK':
            all_click_events.append(event)
        elif event['event'] in ('KEY_PRESS', 'KEY_RELEASE'):
            all_keyboard_events.append(event)

    print(f"Found {len([e for e in data['events'] if e['event'] == 'MOUSE' and e['event_type'] == 'SINGLE_CLICK'])} click events in {json_path.name}")
    print(f"Found {len([e for e in data['events'] if e['event'] in ('KEY_PRESS', 'KEY_RELEASE')])} keyboard events in {json_path.name}")

# Sort all events by timestamp
all_events.sort(key=lambda x: datetime.fromisoformat(x['timestamp']))
all_click_events.sort(key=lambda x: datetime.fromisoformat(x['timestamp']))
all_keyboard_events.sort(key=lambda x: datetime.fromisoformat(x['timestamp']))

print(f"Total events across all files: {len(all_events)}")
print(f"Total click events: {len(all_click_events)}")
print(f"Total keyboard events: {len(all_keyboard_events)}")

# Initialize the history buffer for valid click+screenshot pairs
valid_click_history = deque(maxlen=MAX_HISTORY_ITEMS)

# Now match all click events to screenshots and session prompts
screenshot_idx = 0
keyboard_idx = 0  # Index to keep track of our position in the keyboard events list
last_valid_screenshot = None
skipped_events = 0

for click_event in all_click_events:
    click_time = datetime.fromisoformat(click_event['timestamp'])

    # Match with screenshot
    # Move screenshot pointer forward as long as the screenshot time is <= click time
    while (screenshot_idx < len(images_with_timestamps) and
          images_with_timestamps[screenshot_idx][0] <= click_time):
        last_valid_screenshot = images_with_timestamps[screenshot_idx]
        screenshot_idx += 1

    # For session prompts, since timestamps represent end of session,
    # find which session this click belongs to by finding the NEXT session timestamp
    # after the click time
    session_idx = 0
    while session_idx < len(session_prompts) and session_prompts[session_idx]['timestamp'] <= click_time:
        session_idx += 1

    # The session prompt for this click is the one before the next session
    # (which means this click belongs to the session that ends at session_prompts[session_idx-1])
    current_session = session_prompts[session_idx-1] if session_idx > 0 else None

    if last_valid_screenshot:
        # Calculate time difference in seconds
        time_diff = (click_time - last_valid_screenshot[0]).total_seconds()

        # Only add to dataset if screenshot is within the allowed time range
        if time_diff <= MAX_TIME_DIFF_SECONDS:
            # Get full path to the screenshot image
            screenshot_path = str(last_valid_screenshot[1])

            # Create the basic match entry
            match = {
                "click_timestamp": click_time.isoformat(),
                "click_x": click_event["x"],
                "click_y": click_event["y"],
                "matched_screenshot_timestamp": last_valid_screenshot[0].isoformat(),
                "matched_screenshot_filename": last_valid_screenshot[1].name,
                "screenshot_image": screenshot_path,  # Store full path for HFImage feature
                "source_file": click_event["source_file"],
                "time_diff_seconds": time_diff
            }

            # Add session prompt if available
            if current_session:
                match["session_end_timestamp"] = current_session['timestamp'].isoformat()
                match["session_prompt"] = current_session['prompt']
            else:
                match["session_end_timestamp"] = ""
                match["session_prompt"] = ""

            # Add historical context
            # First determine how many historical items we have
            history_count = len(valid_click_history)
            match["history_count"] = history_count

            # Add historical clicks and screenshots
            for i, (hist_click, hist_screenshot) in enumerate(valid_click_history):
                # Add historical click data
                match[f"history_click_{i}_timestamp"] = hist_click["timestamp"]
                match[f"history_click_{i}_x"] = hist_click["x"]
                match[f"history_click_{i}_y"] = hist_click["y"]

                # Add historical screenshot data
                match[f"history_screenshot_{i}_timestamp"] = hist_screenshot[0].isoformat()
                match[f"history_screenshot_{i}_filename"] = hist_screenshot[1].name
                match[f"history_screenshot_{i}_image"] = str(hist_screenshot[1])

            # Fill in empty values for missing historical items
            for i in range(history_count, MAX_HISTORY_ITEMS):
                match[f"history_click_{i}_timestamp"] = ""
                match[f"history_click_{i}_x"] = 0.0
                match[f"history_click_{i}_y"] = 0.0
                match[f"history_screenshot_{i}_timestamp"] = ""
                match[f"history_screenshot_{i}_filename"] = ""
                match[f"history_screenshot_{i}_image"] = None  # Use None for missing screenshots

            # Determine time range for relevant keyboard events
            if history_count > 0:
                oldest_time = datetime.fromisoformat(valid_click_history[0][0]["timestamp"])
            else:
                # If no history, just use a short window before current click
                oldest_time = click_time - timedelta(seconds=5)

            # Collect keyboard events between oldest_time and click_time using sliding window
            # First, move keyboard_idx forward until we find events after oldest_time
            while (keyboard_idx < len(all_keyboard_events) and
                  datetime.fromisoformat(all_keyboard_events[keyboard_idx]["timestamp"]) < oldest_time):
                keyboard_idx += 1

            # Now collect all keyboard events from keyboard_idx until click_time
            relevant_keyboard_events = []
            temp_idx = keyboard_idx
            while (temp_idx < len(all_keyboard_events) and
                  datetime.fromisoformat(all_keyboard_events[temp_idx]["timestamp"]) < click_time):
                relevant_keyboard_events.append(all_keyboard_events[temp_idx])
                temp_idx += 1

            # Process keyboard events into a text representation
            keyboard_text = []
            for event in relevant_keyboard_events:
                if event.get('event') == 'KEY_PRESS':
                    key = event.get('key', '')
                    # Only add printable characters or readable key names
                    if len(key) == 1 or key.startswith('Key.'):
                        keyboard_text.append(key)

            match["keyboard_text"] = ' '.join(keyboard_text)
            match["keyboard_event_count"] = len(relevant_keyboard_events)

            # Add the match to our dataset
            all_matches.append(match)

            # Update the history with this new valid pair
            valid_click_history.append((click_event, last_valid_screenshot))

            print(f"Click at {click_time} matched with screenshot taken at {last_valid_screenshot[0]} (diff: {time_diff:.3f}s)")
            print(f"  Historical context: {history_count} previous events, {len(relevant_keyboard_events)} keyboard events")

            if current_session:
                print(f"  Session: '{current_session['prompt']}' (ends at {current_session['timestamp']})")
            else:
                print("  No session prompt found for this click")
        else:
            print(f"Skipping click at {click_time}: Screenshot too old ({time_diff:.3f}s > {MAX_TIME_DIFF_SECONDS}s)")
            skipped_events += 1
    else:
        print(f"No screenshot before click at {click_time}")
        skipped_events += 1

print(f"Total matches: {len(all_matches)}")
print(f"Skipped events: {skipped_events} (no screenshot within {MAX_TIME_DIFF_SECONDS}s)")

# Create output directory if it doesn't exist
output_path = Path('../data/processed/hf_click_image_dataset_with_history')
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.mkdir(exist_ok=True)

# Define features schema with Image type
features = Features({
    # Current click and screenshot
    "click_timestamp": Value("string"),
    "click_x": Value("float"),
    "click_y": Value("float"),
    "matched_screenshot_timestamp": Value("string"),
    "matched_screenshot_filename": Value("string"),
    "screenshot_image": HFImage(),  # This will load the image from the path
    "source_file": Value("string"),
    "time_diff_seconds": Value("float"),
    "session_end_timestamp": Value("string"),
    "session_prompt": Value("string"),

    # Historical context metadata
    "history_count": Value("int32"),
    "keyboard_event_count": Value("int32"),
    "keyboard_text": Value("string"),
})

# Add historical click and screenshot features
for i in range(MAX_HISTORY_ITEMS):
    # Historical click features
    features[f"history_click_{i}_timestamp"] = Value("string")
    features[f"history_click_{i}_x"] = Value("float")
    features[f"history_click_{i}_y"] = Value("float")

    # Historical screenshot features
    features[f"history_screenshot_{i}_timestamp"] = Value("string")
    features[f"history_screenshot_{i}_filename"] = Value("string")
    features[f"history_screenshot_{i}_image"] = HFImage()

# Convert to Hugging Face Dataset with the defined features
hf_dataset = Dataset.from_list(all_matches, features=features)

# Save to disk
hf_dataset.save_to_disk(output_path)
print(f"Dataset saved to disk with actual images and historical context at {output_path}")